# 08 | Full Development-Cohort Audit

**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**Objective:** Examine all 29 official HARTH v2.0 participants outside the original S023/S032 reserved Test set. Audit label availability and possible GroupKFold partitions **before** more expensive training. No model is fitted and Test data are not loaded.

**Download note:** 19 additional source CSVs total approximately 775 MiB, so review the inventory before permitting downloads. Cached data stay local and Git-ignored.

## 01. Imports and dataset inventory

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Import the existing study adapter and shared group-coverage audit.
root = Path.cwd().resolve()
while not (root / "code/python/src/core/data_loading.py").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook from within the repository.")
    root = root.parent
sys.path.insert(0, str(root / "code/python/src"))
sys.path.insert(0, str(root / "studies/wearable_activity_recognition/src"))
from harth_data import HARTH_SUBJECTS, ACTIVITY_NAMES, prepare_harth_feature_cohort
from core.data_splitting import audit_group_class_coverage
print("Project:", root)

## 02. Inspect the download requirements before starting

The two previously reserved Test subjects are excluded by identifier. Run this cell and check the file inventory. The next cell performs downloads; it is deliberately separate.

In [ ]:
# Freeze original Test IDs; never add them to the development cohort.
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TASK_CLASSES = tuple(range(1, 9))
data_dir = root / "data/raw/harth_v2"
inventory = pd.DataFrame([
    {"subject": subject,
     "locally_cached": (data_dir / f"{subject}.csv").is_file(),
     "local_MiB": round((data_dir / f"{subject}.csv").stat().st_size / 1048576, 2)
     if (data_dir / f"{subject}.csv").is_file() else None}
    for subject in DEVELOPMENT_SUBJECTS
])
assert len(DEVELOPMENT_SUBJECTS) == 29
assert not (set(DEVELOPMENT_SUBJECTS) & RESERVED_TEST)
display(inventory)
print("Development participants:", len(DEVELOPMENT_SUBJECTS))
print("Files not yet cached:", int((~inventory["locally_cached"]).sum()))
print("No source files were downloaded by the inventory cell.")

## 03. Acquire and prepare the development data

**This cell may download approximately 775 MiB**, depending on what is already cached, and requires network access. It processes each person's file independently, keeps only label-pure 2-second windows and applies the same 30-feature methods as earlier notebooks. Undefined source codes are reported rather than reinterpreted.

In [ ]:
# Reuse the study adapter; filtering occurs AFTER time-safe windowing.
features, metadata, overview, excluded = prepare_harth_feature_cohort(
    subjects=DEVELOPMENT_SUBJECTS,
    data_dir=data_dir,
    task_classes=TASK_CLASSES,
    window_size=100,
    step_size=50,
    sampling_rate=50,
)
display(overview)
print("Development feature matrix:", features.shape)
print("Excluded source codes:")
display(excluded.sort_values(["source_code", "subject_id"]))

## 04. Validate participant/activity coverage and fold feasibility

GroupKFold is deterministic and participant-disjoint, but cannot promise class-balanced validation. This is a **pre-training audit**, not model evaluation.

In [ ]:
# Count each documented activity independently for every participant.
coverage, folds = audit_group_class_coverage(
    labels=metadata["label"].to_numpy(),
    group_ids=metadata["subject_id"].to_numpy(),
    classes=TASK_CLASSES,
    n_splits=5,
)
coverage.columns = [f"{code}: {ACTIVITY_NAMES[code]}" for code in TASK_CLASSES]
display(coverage)
display(folds)
print("Missing training classes by fold:",
      folds[["fold", "missing_train_classes"]].to_dict("records"))
print("Missing validation classes by fold:",
      folds[["fold", "missing_validation_classes"]].to_dict("records"))

## 05. Visualise actual coverage

The heatmap displays the number of eligible windows by person and activity. A separate chart shows the proportion of persons contributing to each activity; large window counts must not be confused with independent subject support.

In [ ]:
# 1. Visualise person-by-class counts using the original units (windows).
fig, axes = plt.subplots(2, 1, figsize=(14, 14))
matrix = coverage.to_numpy()
heat = axes[0].imshow(matrix, aspect="auto", cmap="YlGnBu")
axes[0].set_xticks(np.arange(coverage.shape[1]), coverage.columns,
                   rotation=40, ha="right")
axes[0].set_yticks(np.arange(coverage.shape[0]), coverage.index)
axes[0].set(title="HARTH v2.0: development subject/class coverage",
            xlabel="Documented activity", ylabel="Participant")
fig.colorbar(heat, ax=axes[0], label="Eligible windows")

# 2. Count independent subjects, not windows, for each documented class.
subject_support = (coverage > 0).sum(axis=0)
axes[1].barh(subject_support.index, subject_support.to_numpy())
axes[1].set(title="Independent development participants per activity",
            xlabel="Number of participants", xlim=(0, len(DEVELOPMENT_SUBJECTS)))
axes[1].grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

## 06. Quality gate and next decision

Confirm that all planned development IDs have been processed and reserved Test IDs excluded. Then decide on a full-cohort grouped-CV protocol and preprocessing policy before training further models.

In [ ]:
# Verify subjects, feature integrity and fold membership in the audit.
assert metadata["subject_id"].nunique() == 29
assert not (set(metadata["subject_id"]) & RESERVED_TEST)
assert features.shape == (len(metadata), 30)
assert np.isfinite(features.to_numpy()).all()
assert len(folds) == 5
assert sum(overview["eligible_windows"]) == len(metadata)
assert sum(overview["excluded_windows"]) == sum(excluded["excluded_windows"])
print("Full development-cohort audit: PASSED")
print("Reserved Test participants used: NO")
print("Group-independent model fitting completed: NO")
print("Next: review class coverage, choose grouped evaluation, then train.")